websocket connection demo 

In [2]:
import json
import websockets

URL = "ws://2e4e907b.fast.gemini.com"

async def demo(n=10):
    async with websockets.connect(URL) as ws:
        await ws.send(json.dumps({"id":"1","method":"subscribe","params":["btcusd@bookTicker"]}))
        for _ in range(n):
            print(await ws.recv())

await demo(10)


{"u":1764528176106361,"E":1772396991176341793,"s":"btcusd","b":"65411.79000","B":"0.0034000000","a":"65413.89000","A":"0.0034000000"}
{"id":"1","status":200}
{"u":1764528176106648,"E":1772396991598260428,"s":"btcusd","b":"65405.25000","B":"0.0034000000","a":"65413.89000","A":"0.0034000000"}
{"u":1764528176106716,"E":1772396991606043465,"s":"btcusd","b":"65405.25000","B":"0.0034000000","a":"65412.80000","A":"0.0401344400"}
{"u":1764528176106718,"E":1772396991606177598,"s":"btcusd","b":"65405.25000","B":"0.0034000000","a":"65407.79000","A":"0.0034000000"}
{"u":1764528176106749,"E":1772396991609198021,"s":"btcusd","b":"65400.00000","B":"0.0009645200","a":"65407.79000","A":"0.0034000000"}
{"u":1764528176106750,"E":1772396991609302827,"s":"btcusd","b":"65403.81000","B":"0.0034000000","a":"65407.79000","A":"0.0034000000"}
{"u":1764528176106826,"E":1772396991619720016,"s":"btcusd","b":"65400.00000","B":"0.0009645200","a":"65407.79000","A":"0.0034000000"}
{"u":1764528176106827,"E":177239699161

Gemini public api:

In [3]:
# Tradable symbols
import requests
BASE_URL = "https://api.gemini.com"  # or sandbox
symbols = requests.get(BASE_URL + "/v1/symbols", timeout=10).json()
symbols[:20]


['2zgusd',
 '2zrlusd',
 '2zusd',
 '2zusdc',
 'aavegusd',
 'aaverlusd',
 'aaveusd',
 'aaveusdc',
 'aligusd',
 'alirlusd',
 'aliusd',
 'aliusdc',
 'ampgusd',
 'amprlusd',
 'ampusd',
 'ampusdc',
 'ankrgusd',
 'ankrrlusd',
 'ankrusd',
 'ankrusdc']

In [4]:
#trade constraints
sym = "btcusd"
details = requests.get(BASE_URL + f"/v1/symbols/details/{sym}", timeout=10).json()
details


{'symbol': 'BTCUSD',
 'base_currency': 'BTC',
 'quote_currency': 'USD',
 'tick_size': 1e-08,
 'quote_increment': 0.01,
 'min_order_size': '0.00001',
 'status': 'open',
 'wrap_enabled': False,
 'product_type': 'spot',
 'contract_type': 'vanilla',
 'contract_price_currency': 'USD'}

In [5]:
t2 = requests.get(BASE_URL + f"/v2/ticker/{sym}", timeout=10).json()
t2


{'symbol': 'BTCUSD',
 'open': '66279.9',
 'high': '68187.75',
 'low': '65741.47',
 'close': '66960.06',
 'changes': ['65962.71',
  '66279.9',
  '66006.18',
  '66184.16',
  '66848.38',
  '66863',
  '66981.72',
  '66348.58',
  '66452.29',
  '66455.02',
  '66379.99',
  '66537',
  '67139.86',
  '66934.75',
  '67312.8',
  '67343.7',
  '67509.4',
  '67669.13',
  '67323.8',
  '66835.09',
  '66998.39',
  '67154.67',
  '66759.74',
  '66960.06'],
 'bid': '65395.62',
 'ask': '65403.54'}

In [6]:
#order book
book = requests.get(BASE_URL + f"/v1/book/{sym}", params={"limit_bids":10,"limit_asks":10}, timeout=10).json()
book["bids"][0], book["asks"][0]


({'price': '65377.34', 'amount': '0.0196', 'timestamp': '1772396995'},
 {'price': '65377.35', 'amount': '0.0034', 'timestamp': '1772396995'})

In [7]:
trades = requests.get(BASE_URL + f"/v1/trades/{sym}", params={"limit_trades":50}, timeout=10).json()
trades[0]


{'timestamp': 1772396995,
 'timestampms': 1772396995104,
 'tid': 2840140985886181,
 'price': '65377.34',
 'amount': '0.0196',
 'exchange': 'gemini',
 'type': 'sell'}

In [8]:
candles = requests.get(BASE_URL + f"/v2/candles/{sym}/1m", timeout=10).json()
candles[0]  # array row


[1772396880000, 65463.28, 65463.28, 65410.04, 65419.41, 0.00767343]

Private API (we trade via this)

In [9]:
import requests

BASE_URL = "https://api.gemini.com"  # or sandbox

def get_json(path: str, params=None):
    r = requests.get(BASE_URL + path, params=params, timeout=15)
    r.raise_for_status()
    return r.json()

symbols = get_json("/v1/symbols")
print("num symbols:", len(symbols), "example:", symbols[:10])

# pick one symbol from /v1/symbols (Gemini symbols are typically lowercase like "btcusd")
sym = "btcusd"

ticker = get_json(f"/v1/pubticker/{sym}")
book   = get_json(f"/v1/book/{sym}", params={"limit_bids": 10, "limit_asks": 10})
candles = get_json(f"/v2/candles/{sym}/1m")

print("ticker:", ticker)
print("best bid/ask:", book["bids"][0], book["asks"][0])
print("first candle row:", candles[0] if candles else None)


num symbols: 451 example: ['2zgusd', '2zrlusd', '2zusd', '2zusdc', 'aavegusd', 'aaverlusd', 'aaveusd', 'aaveusdc', 'aligusd', 'alirlusd']
ticker: {'bid': '65341.91', 'ask': '65341.99', 'last': '65348.89', 'volume': {'BTC': '290.20900677', 'USD': '18964836.4604219853', 'timestamp': 1772396999000}}
best bid/ask: {'price': '65341.91', 'amount': '0.0396', 'timestamp': '1772396999'} {'price': '65341.92', 'amount': '0.371465', 'timestamp': '1772396999'}
first candle row: [1772396880000, 65463.28, 65463.28, 65410.04, 65419.41, 0.00767343]


In [63]:
import os, time, json, base64, hmac, hashlib, requests
from typing import Optional, Dict, Any

BASE_URL = "https://api.gemini.com"  # or sandbox
os.environ["GEMINI_API_KEY"] = "account-becUzqtHQIEmAkEQz9oA"      # put your key
os.environ["GEMINI_API_SECRET"] = "2DVi7YrDT4np721kMs9Q6yctnXeX"       # put your secret

API_KEY = os.environ["GEMINI_API_KEY"]
API_SECRET = os.environ["GEMINI_API_SECRET"].encode()
import time
_last_nonce = 0

def next_nonce_seconds():
    global _last_nonce
    while True:
        n = int(time.time())      # seconds
        if n > _last_nonce:
            _last_nonce = n
            return str(n)
        time.sleep(0.05)

def gemini_private_post(path: str, payload_extra: Optional[Dict[str, Any]] = None):
    payload = {
    "request": path,
    "nonce": next_nonce_seconds(),
}
    if payload_extra:
        payload.update(payload_extra)

    encoded = json.dumps(payload).encode()
    b64 = base64.b64encode(encoded)              # bytes
    sig = hmac.new(API_SECRET, b64, hashlib.sha384).hexdigest()

    headers = {
        "Content-Type": "text/plain",
        "Content-Length": "0",
        "X-GEMINI-APIKEY": API_KEY,
        "X-GEMINI-PAYLOAD": b64.decode(),
        "X-GEMINI-SIGNATURE": sig,
        "Cache-Control": "no-cache",
    }
     # ... your existing signing code that builds headers ...
    r = requests.post(BASE_URL + path, headers=headers, timeout=15)
    if not r.ok:
        print("STATUS:", r.status_code)
        print("BODY:", r.text)   # <-- THIS
        r.raise_for_status()
    return r.json()



In [64]:
balances = gemini_private_post("/v1/balances", {
    "showPendingBalances": True
})
print(balances[:2])


[{'type': 'exchange', 'currency': 'USD', 'amount': '2989.980141451', 'available': '2989.980141451', 'availableForWithdrawal': '2989.980141451'}, {'type': 'exchange', 'currency': 'BTC', 'amount': '0.00014839', 'available': '0.00014839', 'availableForWithdrawal': '0.00014839'}]


Order placement

In [65]:
import requests

BASE_URL = "https://api.gemini.com"  # or "https://api.sandbox.gemini.com"
sym = "btcusd"

t = requests.get(f"{BASE_URL}/v1/pubticker/{sym}", timeout=10).json()
ask = float(t["ask"])          # buy near ask
usd_notional = 10.0
amount_btc = usd_notional / ask
import requests

BASE_URL = "https://api.gemini.com"
# BASE_URL = "https://api.sandbox.gemini.com"

def gemini_public_get(path: str, params=None):
    r = requests.get(BASE_URL + path, params=params or {}, timeout=20)
    r.raise_for_status()
    return r.json()
book = gemini_public_get(f"/v1/book/{sym}", params={"limit_bids": 1, "limit_asks": 1})
best_bid = float(book["bids"][0]["price"])
best_ask = float(book["asks"][0]["price"])
print("best_bid, best_ask:", best_bid, best_ask)
details = gemini_public_get(f"/v1/symbols/details/{sym}")



best_bid, best_ask: 65353.99 65355.04


In [ ]:
import time
# Place buy order
order = gemini_private_post("/v1/order/new", {
    "symbol": sym,
    "amount": f"{0.00014839:.8f}",          # BTC qty (string)
    "price":  f"{best_bid*0.998:.2f}",            # cross the spread to fill quickly
    "side":   "sell",
    "type": "exchange limit", #limit orders only execute at the specified price or better, while market orders execute immediately at the best available price. Market orders do not have a price field.
    "options": ["immediate-or-cancel"],     # IOC
    "client_order_id": f"long100-{int(time.time())}",
})
print(order)
print("is_cancelled:", order.get("is_cancelled"))
print("executed_amount:", order.get("executed_amount"))
print("remaining_amount:", order.get("remaining_amount"))
print("avg_execution_price:", order.get("avg_execution_price"))

{'order_id': '73771273540545653', 'id': '73771273540545653', 'symbol': 'btcusd', 'exchange': 'gemini', 'avg_execution_price': '65465.92', 'side': 'sell', 'type': 'exchange limit', 'timestamp': '1772399502', 'timestampms': 1772399502074, 'is_live': False, 'is_cancelled': False, 'is_hidden': False, 'was_forced': False, 'executed_amount': '0.00014839', 'client_order_id': 'long100-1772399502', 'options': ['immediate-or-cancel'], 'price': '65223.28', 'original_amount': '0.00014839', 'remaining_amount': '0'}
is_cancelled: False
executed_amount: 0.00014839
remaining_amount: 0
avg_execution_price: 65465.92
